# Lab 04 — Attention & Memory Retrieval

*Neural Memory Reasoning · MemN2N · PyTorch*

**Goal:** Use the memory vectors from Lab 03 to **score**, **attend to**, and **retrieve from** story memory slots.

**How to work:** Read one challenge, use its provided inputs, and write your own code in the empty cell. **No solutions are included.** Only the input-data cells contain code.

| Stage | Operation | Output |
|---|---|---|
| 1 · Scores | Compare question `u` with input memories `m` | `s` — `(2, 2)` |
| 2 · Attention | Softmax over the memory slots | `p` — `(2, 2)` |
| 3 · Retrieval | Weighted sum of output memories `c` | `o` — `(2, 4)` |
| 4 · Apply | Change queries, mask empty slots, build a reusable read function | Same core shapes |

**Scope:** One attention-and-retrieval step only. No new embedding layers, model training, answer prediction, or multi-hop reasoning in this lab.

---
## Provided Input — Toy MemN2N Vectors

These are **already-aggregated sentence and question vectors**. Think of them as the outputs of `A`, `C` and `B` from Lab 03. Their numbers are deliberately chosen so that attention is easy to inspect; **they are not learned embeddings**.

| Tensor | Meaning | Shape |
|---|---|---|
| `m` | Input memory vectors from `A` | `(2, 2, 4)` |
| `c` | Output memory vectors from `C` | `(2, 2, 4)` |
| `u` | Question vectors from `B` | `(2, 4)` |

Dimensions: **2 stories × 2 memory slots per story × 4 embedding features**. The first and second stories each have one question. Run this **input cell** before Challenge 1.

In [5]:
import torch

# Input memories: batch=2, slots=2, embedding_dim=4
m = torch.tensor([
    [[2., 0., 0., 0.], [0., 2., 0., 0.]],
    [[0., 2., 0., 0.], [0., 0., 2., 0.]],
])

# Output memories: same story / slot layout, different representation
c = torch.tensor([
    [[1., 0., 0., 0.], [0., 1., 0., 0.]],
    [[0., 1., 0., 0.], [0., 0., 1., 0.]],
])

# One question vector per story
u = torch.tensor([
    [1., 0., 0., 0.],
    [0., 0., 1., 0.],
])

print('m:', m.shape, '| c:', c.shape, '| u:', u.shape)

m: torch.Size([2, 2, 4]) | c: torch.Size([2, 2, 4]) | u: torch.Size([2, 4])


---
## Challenge 1 — Attention Scores (Dot Product)

**Objective:** Measure how well each question matches each **input memory slot**.

$$s_{b,i}=u_b^\top m_{b,i}$$

- $b$ = story index; $i$ = memory-slot index.
- Scores are **not probabilities**. A larger score means stronger matching in this toy vector space.

### 1.1 — Score one memory slot

**Input:** `u[0]` and `m[0, 0]`, each shape `(4,)`.

- [ ] Compute their dot product using PyTorch.
- [ ] Print the score and its shape.
- [ ] Explain why a single score is a scalar, rather than a 4D vector.


In [6]:
score = torch.dot(u[0], m[0, 0])

print(score)        
print(score.shape)  

tensor(2.)
torch.Size([])


### 1.2 — Score all memories in a batch

**Input:** `m` `(2, 2, 4)` and `u` `(2, 4)`.

- [ ] Give `u` an extra memory-slot dimension so it can match all slots of its story.
- [ ] Multiply by `m` **element-wise** and sum across the **embedding dimension**.
- [ ] Save the result as `s` and print its shape and values.

**Shape guide:** `u` `(2, 4)` → `(2, 1, 4)` → compare with `m` `(2, 2, 4)` → scores `(2, 2)`.

**Hint:** `.unsqueeze(1)` and `.sum(dim=-1)`. Here `dim=-1` is the **embedding** dimension, not the memory-slot dimension.

**Expected `s`:** first story `[2, 0]`; second story `[0, 2]`.

In [7]:
s = (u.unsqueeze(1) * m).sum(dim=-1)

print(s.shape)  
print(s)

torch.Size([2, 2])
tensor([[2., 0.],
        [0., 2.]])


---
## Challenge 2 — Attention Probabilities (Softmax)

**Objective:** Turn each story's raw scores into a distribution over its memory slots.

$$p_{b,i}=\frac{e^{s_{b,i}}}{\sum_j e^{s_{b,j}}}$$

### 2.1 — Normalize and inspect attention

**Input:** `s` with shape `(2, 2)` from Challenge 1.2.

- [ ] Apply `torch.softmax()` across the **memory-slot dimension**.
- [ ] Store the probabilities in `p`.
- [ ] Print `p` and confirm its shape `(2, 2)`.
- [ ] Check that each story's probabilities sum to `1`.
- [ ] Find the index of the highest-attended slot for each story.

**Hint:** For `(batch, slots)`, slots are at `dim=1`; `.argmax(dim=1)` gives the top slot index.

**Expected approximately:** Story 1 `[0.8808, 0.1192]`; Story 2 `[0.1192, 0.8808]`. Top slot indices: `[0, 1]`.

**Think:** Softmax **weights all valid slots**; it does not hard-select only one slot.

In [8]:
p = torch.softmax(s, dim=1)

print(p)
print(p.shape)
print(p.sum(dim=1))
print(p.argmax(dim=1))

tensor([[0.8808, 0.1192],
        [0.1192, 0.8808]])
torch.Size([2, 2])
tensor([1.0000, 1.0000])
tensor([0, 1])


---
## Challenge 3 — Weighted Memory Retrieval

**Objective:** Retrieve one information vector per story, using the **output memories `c`** rather than the input memories `m`.

$$o_b=\sum_i p_{b,i}\,c_{b,i}$$

### 3.1 — Weight each output memory

**Input:** `p` `(2, 2)` and `c` `(2, 2, 4)`.

- [ ] Add one dimension to `p` for broadcasting.
- [ ] Multiply each output memory vector by the probability of its slot.
- [ ] Name the result `weighted_memories` and inspect its shape.

**Shape guide:** `(2, 2)` → `(2, 2, 1)` → multiply by `(2, 2, 4)` → `(2, 2, 4)`.

**Hint:** `.unsqueeze(-1)` attaches one attention weight to all four features of the same memory slot.

In [9]:
weighted_memories = p.unsqueeze(-1) * c

print(weighted_memories)
print(weighted_memories.shape)

tensor([[[0.8808, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.1192, 0.0000, 0.0000]],

        [[0.0000, 0.1192, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.8808, 0.0000]]])
torch.Size([2, 2, 4])


### 3.2 — Retrieve one vector per story

**Input:** `weighted_memories` `(2, 2, 4)`.

- [ ] Sum across **memory slots only**.
- [ ] Name the result `o` and print its shape and values.
- [ ] Check that the two stories remain separate.

**Expected shape:** `o` `(2, 4)`.

**Expected approximately:** First story `[0.8808, 0.1192, 0, 0]`; second story `[0, 0.1192, 0.8808, 0]`.

**Key distinction:** `m` is used for **matching**; `c` is used for **retrieval**.

In [10]:
o = weighted_memories.sum(dim=1)

print(o)
print(o.shape)

tensor([[0.8808, 0.1192, 0.0000, 0.0000],
        [0.0000, 0.1192, 0.8808, 0.0000]])
torch.Size([2, 4])


---
## Challenge 4 — Sanity Checks & Reusable Memory Read

### 4.1 — Change the question, not the story

**Input:** The original `m`, `c`, `u` and your attention calculations.

- [ ] Copy `u` into `u_changed` without modifying the original tensor.
- [ ] Replace **only the first question** with the vector `[0., 1., 0., 0.]`.
- [ ] Recompute attention probabilities for `u_changed`.
- [ ] Compare with the original `p`.

**Expected:** For Story 1, the **second slot (index 1)** now receives the larger attention weight. Story 2 stays the same.

**Think:** The memories did not change—only the query changed.

In [11]:
u_changed = u.clone()
u_changed[0] = torch.tensor([0., 1., 0., 0.])

s_changed = (u_changed.unsqueeze(1) * m).sum(dim=-1)
p_changed = torch.softmax(s_changed, dim=1)

print(p)
print(p_changed)

tensor([[0.8808, 0.1192],
        [0.1192, 0.8808]])
tensor([[0.1192, 0.8808],
        [0.1192, 0.8808]])


### 4.2 — Ignore an empty (padded) memory slot

Unlike a PAD **word**, a padded **memory slot** is an entire missing sentence. It must receive **zero attention probability**.

**Provided input:** The next cell adds one all-zero memory slot to each story and supplies a boolean slot mask.

| Variable | Shape |
|---|---|
| `m_padded`, `c_padded` | `(2, 3, 4)` |
| `valid_slots` | `(2, 3)` — `True` means real slot |

**Your task:**

- [ ] Compute dot-product scores for the padded memories.
- [ ] Set **invalid slot scores** to negative infinity **before** softmax.
- [ ] Apply softmax over slots; verify the last slot gets probability `0`.
- [ ] Retrieve the output vector; verify it matches `o` from Challenge 3 (allow small floating-point tolerance).

**Hint:** `.masked_fill()` can replace scores where `valid_slots` is `False`. Merely multiplying the output memory by zero is **not enough**: softmax could still assign probability to the missing slot.

**Safety of the math:** Each story has at least one valid slot, so softmax always has a valid denominator.

In [12]:
# Provided input for Challenge 4.2 — not a solution
m_padded = torch.cat([m, torch.zeros(2, 1, 4)], dim=1)
c_padded = torch.cat([c, torch.zeros(2, 1, 4)], dim=1)
valid_slots = torch.tensor([
    [True, True, False],
    [True, True, False],
])

print('m_padded:', m_padded.shape)
print('c_padded:', c_padded.shape)
print('valid_slots:', valid_slots.shape)

m_padded: torch.Size([2, 3, 4])
c_padded: torch.Size([2, 3, 4])
valid_slots: torch.Size([2, 3])


In [13]:
s_padded = (u.unsqueeze(1) * m_padded).sum(dim=-1)
s_padded = s_padded.masked_fill(~valid_slots, float("-inf"))

p_padded = torch.softmax(s_padded, dim=1)
o_padded = (p_padded.unsqueeze(-1) * c_padded).sum(dim=1)

print(p_padded)
print(o_padded)
print(torch.allclose(o_padded, o))

tensor([[0.8808, 0.1192, 0.0000],
        [0.1192, 0.8808, 0.0000]])
tensor([[0.8808, 0.1192, 0.0000, 0.0000],
        [0.0000, 0.1192, 0.8808, 0.0000]])
True


### 4.3 — Create a reusable attention-and-retrieval function

**Objective:** Package the operations you already implemented into **one function** for Lab 05.

**Inputs:** `m`, `c`, `u`, and an **optional** memory-slot mask `valid_slots`.

- [ ] Define a function `memory_read(m, c, u, valid_slots=None)`.
- [ ] Compute dot-product scores `s`.
- [ ] If a slot mask is provided, exclude invalid slots **before softmax**.
- [ ] Compute attention probabilities `p` and retrieved vectors `o`.
- [ ] Return `s`, `p`, `o`.
- [ ] Test it once with `(m, c, u)` and once with `(m_padded, c_padded, u, valid_slots)`.

**Expected without padding:** `s` `(2, 2)` · `p` `(2, 2)` · `o` `(2, 4)`.

**Expected with a padded slot:** `s` `(2, 3)` · `p` `(2, 3)` · `o` `(2, 4)`.

**Note:** It is useful to repeat the *calculation inside a reusable function* here. Don't recreate `A`, `B`, `C` or the tokenization pipeline.

In [14]:
def memory_read(m, c, u, valid_slots=None):
    s = (u.unsqueeze(1) * m).sum(dim=-1)

    if valid_slots is not None:
        s = s.masked_fill(~valid_slots, float("-inf"))

    p = torch.softmax(s, dim=1)
    o = (p.unsqueeze(-1) * c).sum(dim=1)

    return s, p, o


s, p, o = memory_read(m, c, u)
print(s.shape, p.shape, o.shape)

s, p, o = memory_read(m_padded, c_padded, u, valid_slots)
print(s.shape, p.shape, o.shape)

torch.Size([2, 2]) torch.Size([2, 2]) torch.Size([2, 4])
torch.Size([2, 3]) torch.Size([2, 3]) torch.Size([2, 4])


---
## Lab 04 — Completion Checklist

- [ ] I can compute a dot-product **matching score** between `u` and each `mᵢ`.
- [ ] I can apply softmax **across memory slots**, not across embedding features.
- [ ] I can explain why `p.unsqueeze(-1)` is needed for weighted retrieval.
- [ ] I can retrieve `o` as the attention-weighted sum of `c`.
- [ ] I can mask missing **sentence slots** before softmax.
- [ ] My reusable read function works for regular and padded stories.

**One-line summary:** `m + u → scores → softmax probabilities → weighted c → o`.

**Next:** Lab 05 — One-hop and Multi-hop MemN2N. Answer prediction, model training and learned attention come later. This lab uses **illustrative, fixed vectors**; a trained model's attention distribution is not guaranteed to match these examples.